# 보스턴 주택 가격 예측

In [19]:
from tensorflow.keras.datasets.boston_housing import load_data
import numpy as np

# 데이터를 다운받습니다.
(X_train, Y_train), (X_test, Y_test) = load_data(path='boston_housing.npz',
                                                 test_split=0.2,
                                                 seed=777)

print(X_train.shape, Y_train.shape)
print(X_test.shape, Y_test.shape)

(404, 13) (404,)
(102, 13) (102,)


In [20]:
np.mean(X_train, axis = 1).shape

(404,)

# 데이터 전처리

In [21]:
import numpy as np

# 데이터 표준화
# 사이킷런에서 standard scaler 사용해도 됨
mean = np.mean(X_train, axis = 0)
std = np.std(X_train, axis = 0)

x_train = (X_train - mean) / std
x_test = (X_test - mean) / std

# 훈련 데이터셋과 검증 데이터셋으로 나눕니다.
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(x_train, Y_train,
                                                  test_size = 0.33,
                                                  random_state = 777)

# 사이킷런을 이용한 표준화 (X_train, X_test는 위에 없지만 구분하기 위해 사용함)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaler_model = scaler.fit(X_train)
X_train_scaled = scaler_model.transform(X_train)
X_test_scaled = scaler_model.transform(X_test)

# Scaler 모델 저장

In [22]:
import joblib

joblib.dump(scaler_model, 'scaler.pkl')

['scaler.pkl']

# 모델 구성
- 모델의 마지막 Dense 층에서 별도의 활성화 함수를 사용하지 않음
  - 인자를 설정하지 않은 경우, default는 ‘linear’로 설정
- 손실 함수는 회귀 문제에서 주로 사용되는 평균 제곱 오차(MSE; Mean Squared Error)를 사용


In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()

# 회귀를 딥러닝으로 구현할 경우 맨 마지막에 activation function이 존재하지 않는다 O_O
# 입력 데이터의 형태를 꼭 명시해야 합니다.
# 13차원의 데이터를 입력으로 받고, 64개의 출력을 가지는 첫 번째 Dense 층
model.add(Dense(64, activation = 'relu', input_shape = (13, )))
model.add(Dense(32, activation = 'relu')) # 32개의 출력을 가지는 Dense 층
model.add(Dense(1)) # 하나의 값을 출력합니다. -> 최종 가격을 출력해야함

model.compile(optimizer = 'adam', loss = 'mse', metrics = ['mae'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [24]:
history = model.fit(x_train, y_train,
                    epochs = 300,
                    validation_data = (x_val, y_val))

Epoch 1/300
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - loss: 518.0676 - mae: 21.0153 - val_loss: 564.3288 - val_mae: 21.4232
Epoch 2/300
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 487.5148 - mae: 20.2785 - val_loss: 528.6355 - val_mae: 20.5950
Epoch 3/300
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 451.7397 - mae: 19.4250 - val_loss: 486.1513 - val_mae: 19.5815
Epoch 4/300
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 408.9350 - mae: 18.3417 - val_loss: 435.0373 - val_mae: 18.2993
Epoch 5/300
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 357.0922 - mae: 17.0157 - val_loss: 374.4536 - val_mae: 16.6917
Epoch 6/300
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 297.6152 - mae: 15.3639 - val_loss: 306.7700 - val_mae: 14.8146
Epoch 7/300
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 233.8309 - mae: 13.4188 - val_loss: 237.3018 - val_mae: 12.6244
Epoch 8/300
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 172.0105 - mae: 11.1920 - val_loss: 174.4529 - val_mae: 10.5945
Epoch 9/300
9/9 ━━━━━━━━

In [26]:
model.evaluate(x_test, y_test)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 8.7985 - mae: 2.1524 


[8.798545837402344, 2.152358293533325]

In [28]:
print(X_test[0])
print(y_test[0])

[1.6902e-01 0.0000e+00 2.5650e+01 0.0000e+00 5.8100e-01 5.9860e+00
 8.8400e+01 1.9929e+00 2.0000e+00 1.8800e+02 1.9100e+01 3.8502e+02
 1.4810e+01]
21.4


In [30]:
# Scaler 모델 불러오기
import joblib
scaler_load = joblib.load('scaler.pkl')

x_new_data = scaler_load.transform(X_test[0].reshape(1,-1))

print(x_new_data)

[[-0.40993096 -0.48033655  2.05226513 -0.28828791  0.19360146 -0.39363321
   0.67181047 -0.83696465 -0.87754039 -1.3232612   0.30547144  0.32733271
   0.24563206]]


In [32]:
price = model.predict(x_new_data)
print(f"실제 집 값 : {y_test[0]}, 예측한 집 값: {price}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
실제 집 값 : 21.4, 예측한 집 값: [[20.804468]]


# K-Fold 사용하기

# DataSet 다시 로드, 표준화

In [14]:
from tensorflow.keras.datasets.boston_housing import load_data
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import numpy as np

(x_train, y_train), (x_test, y_test) = load_data(path='boston_housing.npz',
                                                 test_split=0.2,
                                                 seed=777)

# 데이터 표준화
mean = np.mean(x_train, axis = 0)
std = np.std(x_train, axis = 0)
# 여기까진 전부 동일합니다.
x_train = (x_train - mean) / std
x_test = (x_test - mean) / std

# K-Fold 정의

In [16]:
from sklearn.model_selection import KFold
#----------------------------------------
# K-Fold를 진행해봅니다.
k = 3

# 주어진 데이터셋을 k만큼 등분합니다.
# 여기서는 3이므로 훈련 데이터셋(404개)를 3등분하여
# 1개는 검증셋으로, 나머지 2개는 훈련셋으로 활용합니다.
kfold = KFold(n_splits=k, random_state = 777, shuffle = True)

# 재사용을 위해 모델을 반환하는 함수를 정의합니다.
# 위에서 사용했던 모델과 동일함
def get_model():
    model = Sequential()
    model.add(Dense(64, activation = 'relu', input_shape = (13, )))
    model.add(Dense(32, activation = 'relu'))
    model.add(Dense(1))

    model.compile(optimizer = 'adam', loss = 'mse', metrics = ['mae'])

    return model

mae_list = [] # 테스트셋을 평가한 후 결과 mae를 담을 리스트를 선언합니다.

# k번 진행합니다.
for train_index, val_index in kfold.split(x_train):
    # 해당 인덱스는 무작위로 생성됩니다.
    # 무작위로 생성해주는 것은 과대적합을 피할 수 있는 좋은 방법입니다.
    x_train_fold, x_val_fold = x_train[train_index], x_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

    # 모델을 불러옵니다.
    model = get_model()

    model.fit(x_train_fold, y_train_fold, epochs = 300, validation_data = (x_val_fold, y_val_fold))

    test_mse, test_mae = model.evaluate(x_test, y_test)
    mae_list.append(test_mae)

Epoch 1/300


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 510.8868 - mae: 20.9164 - val_loss: 552.2876 - val_mae: 21.2997
Epoch 2/300
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 475.6492 - mae: 20.0962 - val_loss: 513.4477 - val_mae: 20.4127
Epoch 3/300
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 437.4030 - mae: 19.1612 - val_loss: 468.3764 - val_mae: 19.3478
Epoch 4/300
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 391.8806 - mae: 18.0012 - val_loss: 416.1714 - val_mae: 18.0467
Epoch 5/300
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 340.1410 - mae: 16.5908 - val_loss: 356.1733 - val_mae: 16.4232
Epoch 6/300
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 283.1090 - mae: 14.9256 - val_loss: 289.8610 - val_mae: 14.4836
Epoch 7/300
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 221.3281 - mae: 12.9653 - val_loss: 222.5389 - val_mae: 12.3377
Epoch 8/300
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 159.7223 - mae: 10.7906 - val_loss: 160.5017 - val_mae: 10.1136
Epoch 9/300
9/9 ━━━━━━━━━━━━━━━━━━━━

In [17]:
print(f'전체 결과: {mae_list}')
print(f'평균낸 결과를 최종 결과로 사용합니다: {np.mean(mae_list)}')

전체 결과: [2.203289031982422, 2.1137993335723877, 2.3151726722717285]
평균낸 결과를 최종 결과로 사용합니다: 2.2107536792755127


# Loss 비교
## 기존 모델의 Loss : 2.277  
## K-Fold 모델의 평균 Loss : 2.210  

## K-Fold를 사용하면 성능이 더 좋다!!!